In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np

In [3]:
from bayesgpt.simulators import NestedModelFamily
from bayesgpt.simulators.benchmarks import StandardDDM, CollapsingBoundDDM, SuperDDM

In [16]:
# Sampling functions for free parameters
def sample_v(n, context=None):
    return np.random.normal(0.5, 0.2, n)

def sample_a(n, context=None):
    return np.random.uniform(0.5, 2.0, n)

def sample_z(n, context=None):
    return np.random.uniform(0.3, 0.7, n)

def sample_tau(n, context=None):
    return np.random.uniform(0.1, 0.5, n)

In [17]:
# Define global parameter space (superset)
param_names = [
    "v",  # drift
    "a",  # boundary
    "z",  # initial bias
    "tau",  # non-decision time
    "sigma",  # noise scale
    "angle",  # collapse rate
    "s_v",  # All 's' are variability
    "s_a",
    "s_z",
    "s_tau",
    "s_sigma",
    "s_angle",
]

In [18]:
model_family = NestedModelFamily(parameter_names=param_names)

In [20]:
# Variant 1: StandardDDM
standard_free_params = {
    "v": sample_v,
    "a": sample_a,
    "z": sample_z,
    "tau": sample_tau,
}
standard_fixed_params = {
    "s_v": 0.0,
    "sigma": 1.0,
}
model_family.add_variant(
    name="standard_ddm",
    model=StandardDDM,
    free_parameters=standard_free_params,
    fixed_parameters=standard_fixed_params,
    num_samples=100,
)

In [21]:
# Variant 2: CollapsingBoundDDM
collapsing_free_params = {
    "v": sample_v,
    "a": sample_a,
    "z": sample_z,
    "tau": sample_tau,
    "angle": lambda batch_size, context=None: np.random.uniform(0.0, 0.1, batch_size),
}
collapsing_fixed_params = {
    "s_v": 0.0,
    "sigma": 1.0,
}
model_family.add_variant(
    name="collapsing_ddm",
    model=CollapsingBoundDDM,
    free_parameters=collapsing_free_params,
    fixed_parameters=collapsing_fixed_params,
    num_samples=1,
)

In [22]:
# Variant 3: SuperDDM with mixture components
super_free_params = {
    "v": sample_v,
    "a": sample_a,
    "z": sample_z,
    "tau": sample_tau,
    "s_v": lambda n, context=None: np.random.uniform(0.0, 0.1, n),
    "s_z": lambda n, context=None: np.random.uniform(0.0, 0.05, n),
    "s_tau": lambda n, context=None: np.random.uniform(0.0, 0.05, n),
}
super_fixed_params = {
    "sigma": 1.0,
    "angle": 0.0,
}
model_family.add_variant(
    name="super_ddm",
    model=SuperDDM,
    free_parameters=super_free_params,
    fixed_parameters=super_fixed_params,
    num_samples=1,
)

In [23]:
# Simulate from each variant
batch_size = 5
for variant_name in model_family.variant_names:
    print(f"\nSimulating from {variant_name}:")
    result = model_family.sample(variant_name=variant_name, batch_size=batch_size)
    print("Simulation data (rts, choices):", result["sim_data"])
    print("Full parameters:", result["full_params"])
    print("Inference conditions:", result["inference_conditions"])


Simulating from standard_ddm:


TypeError: StandardDDM.simulate() takes from 2 to 3 positional arguments but 4 were given

In [12]:
# Print shapes of all data
for variant, result in results.items():
    print(f"\nShapes for {variant}:")
    print("  sim_data:")
    for key, value in result['sim_data'].items():
        print(f"    {key}: {value.shape}")
    print(f"  full_params: {result['full_params'].shape}")
    print(f"  inference_conditions: {result['inference_conditions'].shape}")

# Display sample results
for variant, result in results.items():
    print(f"\nResults for {variant}:")
    print(f"  First 5 RTs: {result['sim_data']['rts'][:5]}")
    print(f"  First 5 Choices: {result['sim_data']['choices'][:5]}")


Shapes for std_ddm_1:
  sim_data:
    rts: (100,)
    choices: (100,)
  full_params: (100, 14)
  inference_conditions: (100, 46)

Shapes for coll_ddm_1:
  sim_data:
    rts: (100,)
    choices: (100,)
  full_params: (100, 16)
  inference_conditions: (100, 52)

Shapes for super_ddm_1:
  sim_data:
    rts: (100,)
    choices: (100,)
  full_params: (100, 19)
  inference_conditions: (100, 61)

Shapes for super_ddm_2:
  sim_data:
    rts: (100,)
    choices: (100,)
  full_params: (100, 22)
  inference_conditions: (100, 70)

Results for std_ddm_1:
  First 5 RTs: [1.132 0.849 0.703 0.429 0.493]
  First 5 Choices: [0. 1. 1. 1. 1.]

Results for coll_ddm_1:
  First 5 RTs: [1.245 1.264 1.533 2.619 0.943]
  First 5 Choices: [1. 0. 1. 1. 1.]

Results for super_ddm_1:
  First 5 RTs: [1.265 2.918 2.595 1.582 2.048]
  First 5 Choices: [0. 1. 1. 1. 1.]

Results for super_ddm_2:
  First 5 RTs: [2.188 1.451 2.175 3.57  1.544]
  First 5 Choices: [1. 1. 1. 1. 1.]


In [22]:
# Demonstrate clearing all variants
model_family.remove_all_variants()
print("\nVariants after clearing:", model_family.variant_names)


Variants after clearing: []
